# Story 1 — MMD² as a predictor of monitoring data need

**Pilot: Iceland → Switzerland**

> *Without fine-tuning: RMSE scales with MMD²(source, target bin).*  
> *With fine-tuning: the data needed to close the gap scales with MMD².*  
> *Therefore: MMD² tells you how much monitoring data you need before the model becomes reliable.*

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../'))

from copy import deepcopy

import warnings
import massbalancemachine as mbm
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from sklearn.preprocessing import StandardScaler
from scipy import stats
from tqdm.auto import tqdm
from pathlib import Path
import hashlib, json
import statsmodels.formula.api as smf

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')


## Config

Points to the same cached results as experiment 9 (ISL→CH, original distance-to-Iceland binning). We only READ from this cache — no retraining.

In [ ]:
EXPERIMENT_NUMBER = 9
MODEL = "AdapterLSTM"

BASE_DIR = Path(cfg.dataPath) / path_cache / f"{MODEL}_{EXPERIMENT_NUMBER}"
MODELS_DIR = BASE_DIR / "models"
CACHE_DIR = BASE_DIR / "cache"
RESULTS_DIR = BASE_DIR / "results"
STORY1_DIR = BASE_DIR / "story1"
STORY1_DIR.mkdir(parents=True, exist_ok=True)

MONTHLY_COLS = [
    "t2m", "tp", "slhf", "sshf", "ssrd", "fal", "str", "ELEVATION_DIFFERENCE"
]
STATIC_COLS = ["aspect", "slope", "svf"]
FEATURE_COLS = MONTHLY_COLS + STATIC_COLS

REGION_CODE_MAP = {
    "CH": "11_CH",
    "NOR": "08_NOR",
    "ISL": "06_ISL",
    "CEU": "11_CEU"
}
REGION_GROUPS = {"CEU": ["FR", "CH", "IT_AT"]}


def split_signature(glaciers):
    g = sorted(map(str, glaciers))
    return hashlib.md5((",".join(g)).encode()).hexdigest()[:10]


## Load data and pre-trained assets

In [ ]:
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_Europe.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_Europe.nc"),
}
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"{key} not found: {path}")

dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

vois_climate = ["t2m", "tp", "slhf", "sshf", "ssrd", "fal", "str"]
vois_topographical = ["aspect", "slope", "svf"]

res_xreg, split_info = prepare_monthly_df_xreg_pairwise(
    cfg=cfg,
    dfs=dfs,
    paths=paths,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    source_code="ISL",
    target_code="CH",
    run_flag=False,
    region_name="XREG_ISL_TO_CH",
    csv_subfolder="CrossRegional/ISL_to_CH/csv",
)
# shorthand used throughout
res = res_xreg
df_iceland = res["df_train"]
df_ch_all = res["df_test"]

print(
    f"Iceland : {len(df_iceland)} rows | {df_iceland['GLACIER'].nunique()} glaciers"
)
print(
    f"CH      : {len(df_ch_all)} rows  | {df_ch_all['GLACIER'].nunique()} glaciers"
)


In [ ]:
# Reproduce the same holdout split as experiment 9
best_split, best_score = None, np.inf

for s in range(100):
    split = split_pool_holdout(
        df_region=df_ch_all,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        holdout_frac=0.20,
        seed=s,
    )
    actual_frac = split["actual_holdout_frac"]
    if not (0.15 <= actual_frac <= 0.35):
        continue
    score = abs(split["mmd2_holdout_vs_region"]) + abs(
        split["mmd2_pool_vs_region"])
    if score < best_score:
        best_score, best_split = score, split

holdout_glaciers = best_split["holdout_glaciers"]
pool_glaciers = best_split["pool_glaciers"]
df_holdout = df_ch_all[df_ch_all["GLACIER"].isin(holdout_glaciers)].copy()
df_pool = df_ch_all[df_ch_all["GLACIER"].isin(pool_glaciers)].copy()


In [ ]:
print(f"Holdout : {len(holdout_glaciers)} glaciers | {len(df_holdout)} rows")
print("Holdout glaciers:\n", holdout_glaciers)
print(f"Pool    : {len(pool_glaciers)} glaciers | {len(df_pool)} rows")
print("Pool glaciers:\n", pool_glaciers)

In [ ]:
# Load pre-trained ISL baseline model from experiment 9
sig = split_signature(holdout_glaciers)

static_assets = build_static_tl_assets_source_and_holdout(
    cfg=cfg,
    res_xreg=res_xreg,
    target_codes=["CH"],
    source_codes=["ISL"],
    target_label="CH",
    source_label="ISL",
    holdout_glaciers=holdout_glaciers,
    MONTHLY_COLS=MONTHLY_COLS,
    STATIC_COLS=STATIC_COLS,
    cache_dir=CACHE_DIR / "ISL_to_CH" / f"holdout_{sig}",
    force_recompute=False,
    show_progress=False,
)

BASE_GS_DIR = Path(cfg.dataPath) / path_cache / 'GS_results'
log_path_gs_results = {
    "ISL": BASE_GS_DIR / "lstm_param_search_progress_OOS_ISL_2026-02-11.csv",
    "NOR": BASE_GS_DIR / "lstm_param_search_progress_OOS_NOR_2026-02-09.csv",
    "CH": BASE_GS_DIR / "lstm_param_search_progress_CH_2026-02-18.csv",
}
default_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'loss_name': 'neutral',
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None,
}
params_by_key = build_lstm_params_by_key(
    default_params=default_params,
    log_path_gs_results=log_path_gs_results,
    RGI_REGIONS=RGI_REGIONS,
)
isl_params = params_by_key[REGION_CODE_MAP["ISL"]]

isl_model, _, _ = train_or_load_source_baseline(
    cfg=cfg,
    tl_assets={"STATIC": static_assets},
    default_params=isl_params,
    device=device,
    source_code="ISL",
    models_dir=MODELS_DIR,
    prefix="lstm_ISL",
    key="BASELINE",
    train_flag=False,
    force_retrain=False,
    epochs=150,
)
print("ISL baseline loaded.")


---
## Section 1 — Zero-shot: MMD² predicts generalisation difficulty

**Strategy — sub-region binning within CH:**  
We bin CH glaciers by their MMD² distance to Iceland using `compute_domain_shift`
at glacier level (consistent with the updated distance metric). Each bin is a
coherent subset of CH with a genuinely different domain distance to Iceland.
Evaluating the zero-shot ISL model per bin gives N_bins × (MMD², RMSE) points
from a single pair.


In [ ]:
def measurement_distances_to_source(
    df_pool: pd.DataFrame,
    df_source: pd.DataFrame,
    df_holdout: pd.DataFrame,
    monthly_cols: list[str],
    static_cols: list[str],
    glacier_col: str = "GLACIER",
    id_col: str = "ID",
    seed: int = 0,
    batch_size: int = 500,
) -> pd.DataFrame:
    """
    Compute MMD² distance from each pool measurement (ID) to:
      1) source distribution (Iceland)
      2) holdout distribution (Swiss holdout)

    Climate and topo distances are computed separately with independent
    scalers and bandwidths, then combined as a weighted average.

    Returns
    -------
    pd.DataFrame indexed by ID with:
        - mmd2_climate_vs_iceland
        - mmd2_topo_vs_iceland
        - mmd2_joint_vs_iceland
        - mmd2_climate_vs_holdout
        - mmd2_topo_vs_holdout
        - mmd2_joint_vs_holdout
    """

    def build_X(df):
        m = df.groupby(id_col)[monthly_cols].mean()
        s = df.groupby(id_col)[static_cols].first()
        return m.to_numpy(dtype=np.float64), s.to_numpy(
            dtype=np.float64), m.index.tolist()

    X_src_m, X_src_s, _ = build_X(df_source)
    X_pool_m, X_pool_s, pool_ids = build_X(df_pool)
    X_hold_m, X_hold_s, _ = build_X(df_holdout)

    # --- separate scalers for climate and topo ---
    scaler_m = StandardScaler().fit(np.vstack([X_pool_m, X_src_m, X_hold_m]))
    scaler_s = StandardScaler().fit(np.vstack([X_pool_s, X_src_s, X_hold_s]))

    X_src_mz = scaler_m.transform(X_src_m)
    X_pool_mz = scaler_m.transform(X_pool_m)
    X_hold_mz = scaler_m.transform(X_hold_m)

    X_src_sz = scaler_s.transform(X_src_s)
    X_pool_sz = scaler_s.transform(X_pool_s)
    X_hold_sz = scaler_s.transform(X_hold_s)

    def _get_bandwidths(X_pool_z, X_src_z, X_hold_z, seed):
        rng = np.random.default_rng(seed)
        Z = np.vstack([X_pool_z, X_src_z, X_hold_z])
        if Z.shape[0] > 2000:
            Z = Z[rng.choice(Z.shape[0], 2000, replace=False)]
        z2 = np.sum(Z * Z, axis=1, keepdims=True)
        D2 = np.maximum(z2 + z2.T - 2.0 * (Z @ Z.T), 0.0)
        np.fill_diagonal(D2, 0.0)
        sig = float(np.sqrt(0.5 * np.median(D2[D2 > 0])))
        return [sig * 0.5, sig, sig * 2.0]

    # --- separate bandwidths for climate and topo ---
    bw_m = _get_bandwidths(X_pool_mz, X_src_mz, X_hold_mz, seed=seed)
    bw_s = _get_bandwidths(X_pool_sz, X_src_sz, X_hold_sz, seed=seed + 1)

    def _rbf(A, B, sigma):
        a2 = np.sum(A * A, axis=1, keepdims=True)
        b2 = np.sum(B * B, axis=1, keepdims=True).T
        return np.exp(-np.maximum(a2 + b2 - 2.0 * (A @ B.T), 0.0) /
                      (2.0 * sigma**2))

    def compute_Eyy(X, bandwidths):
        n = X.shape[0]
        out = []
        for sigma in bandwidths:
            K = _rbf(X, X, sigma)
            np.fill_diagonal(K, 0.0)
            out.append(float(K.sum() / (n * (n - 1))))
        return out

    Eyy_src_m = compute_Eyy(X_src_mz, bw_m)
    Eyy_hold_m = compute_Eyy(X_hold_mz, bw_m)
    Eyy_src_s = compute_Eyy(X_src_sz, bw_s)
    Eyy_hold_s = compute_Eyy(X_hold_sz, bw_s)

    # --- batched computation ---
    n_pool = X_pool_mz.shape[0]
    n_batches = int(np.ceil(n_pool / batch_size))

    dist_climate_src = np.empty(n_pool)
    dist_climate_hold = np.empty(n_pool)
    dist_topo_src = np.empty(n_pool)
    dist_topo_hold = np.empty(n_pool)

    for b in tqdm(range(n_batches), desc="MMD² distances"):
        start = b * batch_size
        end = min(start + batch_size, n_pool)

        batch_m = X_pool_mz[start:end]
        batch_s = X_pool_sz[start:end]

        d_clim_src = np.zeros(end - start)
        d_clim_hold = np.zeros(end - start)
        d_topo_src = np.zeros(end - start)
        d_topo_hold = np.zeros(end - start)

        for sigma, ey_sm, ey_hm in zip(bw_m, Eyy_src_m, Eyy_hold_m):
            Kxy_src_m = _rbf(batch_m, X_src_mz, sigma)
            Kxy_hold_m = _rbf(batch_m, X_hold_mz, sigma)
            d_clim_src += 1.0 + ey_sm - 2.0 * Kxy_src_m.mean(axis=1)
            d_clim_hold += 1.0 + ey_hm - 2.0 * Kxy_hold_m.mean(axis=1)

        for sigma, ey_ss, ey_hs in zip(bw_s, Eyy_src_s, Eyy_hold_s):
            Kxy_src_s = _rbf(batch_s, X_src_sz, sigma)
            Kxy_hold_s = _rbf(batch_s, X_hold_sz, sigma)
            d_topo_src += 1.0 + ey_ss - 2.0 * Kxy_src_s.mean(axis=1)
            d_topo_hold += 1.0 + ey_hs - 2.0 * Kxy_hold_s.mean(axis=1)

        dist_climate_src[start:end] = d_clim_src / len(bw_m)
        dist_climate_hold[start:end] = d_clim_hold / len(bw_m)
        dist_topo_src[start:end] = d_topo_src / len(bw_s)
        dist_topo_hold[start:end] = d_topo_hold / len(bw_s)

    return pd.DataFrame({
        "ID":
        pool_ids,
        "mmd2_climate_vs_iceland":
        dist_climate_src,
        "mmd2_topo_vs_iceland":
        dist_topo_src,
        "mmd2_joint_vs_iceland":
        0.5 * dist_climate_src + 0.5 * dist_topo_src,
        "mmd2_climate_vs_holdout":
        dist_climate_hold,
        "mmd2_topo_vs_holdout":
        dist_topo_hold,
        "mmd2_joint_vs_holdout":
        0.5 * dist_climate_hold + 0.5 * dist_topo_hold,
    }).set_index("ID")

In [ ]:
# Compute per-ID distances for all CH measurements
meas_dist = measurement_distances_to_source(
    df_pool=df_ch_all,  # all CH — pool + holdout
    df_source=df_iceland,
    df_holdout=df_holdout,  # still needed for shared scaler
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)

In [ ]:
# Evaluate ISL zero-shot model on all CH, get per-ID predictions
_, df_preds, _, _ = evaluate_one_model_TL(
    cfg=cfg,
    model=isl_model,
    device=device,
    tl_assets_for_key=static_assets,  # uses full CH holdout as test set
    ax=None,
    title=None,
    batch_size=128,
    domain_vocab=static_assets.get("domain_vocab", None),
    show_plot=False,
)
# df_preds should have ID, predicted, observed columns
# compute per-ID absolute error
df_preds["abs_error"] = (df_preds["pred"] - df_preds["target"]).abs()
df_preds["sq_error"] = (df_preds["pred"] - df_preds["target"])**2

In [ ]:
# test_metrics, df_preds = model.evaluate_with_preds(
#     device, test_dl, ds_test_copy)
# test_rmse_a, test_rmse_w = test_metrics['RMSE_annual'], test_metrics[
#     'RMSE_winter']

# print('Test RMSE annual: {:.3f} | winter: {:.3f}'.format(
#     test_rmse_a, test_rmse_w))

scores_annual, scores_winter = compute_seasonal_scores(df_preds,
                                                       target_col='target',
                                                       pred_col='pred')

fig = plt.figure(figsize=(15, 10))
ax1 = plt.subplot(1, 1, 1)

pred_vs_truth_density(
    ax1,
    df_preds,
    scores_annual,
    add_legend=False,
    palette=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER],
    ax_xlim=(-12, 10),
    ax_ylim=(-12, 10),
)

legend_NN = "\n".join([
    r"$\mathrm{RMSE_a}=%.2f$, $\mathrm{RMSE_w}=%.2f$" %
    (scores_annual["rmse"], scores_winter["rmse"]),
    r"$\mathrm{R^2_a}=%.2f$, $\mathrm{R^2_w}=%.2f$" %
    (scores_annual["R2"], scores_winter["R2"]),
    r"$\mathrm{Bias_a}=%.2f$, $\mathrm{Bias_w}=%.2f$" %
    (scores_annual["Bias"], scores_winter["Bias"]),
])
ax1.text(
    0.02,
    0.98,
    legend_NN,
    transform=ax1.transAxes,
    verticalalignment="top",
    fontsize=20,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.5),
)

In [ ]:
dist_cols = {
    "climate": "mmd2_climate_vs_iceland",
    "topo": "mmd2_topo_vs_iceland",
    "joint": "mmd2_joint_vs_iceland",
}

df_preds_dist = df_preds.set_index("ID").join(meas_dist, how="inner")

seasons = {
    "annual": df_preds_dist[df_preds_dist["PERIOD"] == "annual"],
    "winter": df_preds_dist[df_preds_dist["PERIOD"] == "winter"],
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey="row")

for col_idx, (dist_name, dist_col) in enumerate(dist_cols.items()):
    for row_idx, (season_name, df_s) in enumerate(seasons.items()):
        ax = axes[row_idx, col_idx]
        x = df_s[dist_col].values
        y_pred = df_s["pred"].values
        y_target = df_s["target"].values

        bins = np.quantile(x, np.linspace(0, 1, 20))
        bin_idx_arr = np.digitize(x, bins)
        bin_means_x, bin_rmse, bin_std, bin_counts = [], [], [], []

        for b in range(1, len(bins)):
            mask = bin_idx_arr == b
            if mask.sum() > 5:
                residuals = y_pred[mask] - y_target[mask]
                bin_means_x.append(x[mask].mean())
                bin_rmse.append(np.sqrt(np.mean(residuals**2)))
                bin_std.append(np.std(residuals))  # spread of residuals in bin
                bin_counts.append(mask.sum())

        bin_means_x = np.array(bin_means_x)
        bin_rmse = np.array(bin_rmse)
        bin_std = np.array(bin_std)
        bin_counts = np.array(bin_counts)

        ax.fill_between(bin_means_x,
                        bin_rmse - bin_std,
                        bin_rmse + bin_std,
                        color="#185FA5",
                        alpha=0.15,
                        zorder=2)

        ax.plot(bin_means_x,
                bin_rmse,
                color="#185FA5",
                linewidth=1.5,
                linestyle="--",
                alpha=0.6,
                zorder=3)

        ax.scatter(bin_means_x,
                   bin_rmse,
                   s=bin_counts / bin_counts.max() * 300 + 30,
                   color="#185FA5",
                   alpha=0.8,
                   edgecolors="#2C2C2A",
                   linewidths=0.5,
                   zorder=4)

        r, p = stats.spearmanr(bin_means_x, bin_rmse)
        ax.text(0.03,
                0.97,
                f"Spearman r = {r:.2f}  (p = {p:.3f})",
                transform=ax.transAxes,
                fontsize=9,
                va="top",
                color="#444441")

        ax.set_xlabel(f"MMD² {dist_name} (stake, Iceland)", fontsize=10)
        ax.set_ylabel(f"RMSE {season_name} (m w.e.)" if col_idx == 0 else "",
                      fontsize=10)
        ax.set_title(f"{season_name} — {dist_name}", fontsize=11)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(color="#e0e0e0", linewidth=0.6)
        ax.set_axisbelow(True)

fig.suptitle("Story 1.1 — Per-stake MMD² to Iceland predicts zero-shot RMSE",
             fontsize=13)
fig.tight_layout()
fig.savefig(STORY1_DIR / "story1_zeroshot_per_stake.png",
            dpi=150,
            bbox_inches="tight")
plt.show()